In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import time
import warnings

warnings.filterwarnings('ignore')


class SparseMax(nn.Module):
    def __init__(self, dim=-1):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        x_sorted, _ = torch.sort(x, dim=self.dim, descending=True)
        cumsum = torch.cumsum(x_sorted, dim=self.dim)
        k = torch.arange(1, x.shape[self.dim] + 1, device=x.device, dtype=x.dtype)
        z = 1 + k * x_sorted
        mask = (z > cumsum).float()
        k_z = torch.sum(mask, dim=self.dim, keepdim=True)
        tau = (torch.sum(x_sorted * mask, dim=self.dim, keepdim=True) - 1) / k_z
        output = torch.clamp(x - tau, min=0)
        return output


class LocalAttentionModule(nn.Module):
    def __init__(self, group_size, hidden_dim):
        super().__init__()
        self.local_transform = nn.Sequential(
            nn.Linear(group_size, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, group_size)
        )

    def forward(self, x):
        return self.local_transform(x)


class GlobalAttentionModule(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.global_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        return self.global_transform(x)


class HierarchicalAttentionLayer(nn.Module):
    def __init__(self, input_dim, n_groups=4, local_hidden=32, global_hidden=64):
        super().__init__()
        self.input_dim = input_dim
        self.n_groups = n_groups
        self.group_size = input_dim // n_groups
        self.remaining = input_dim % n_groups

        self.local_attentions = nn.ModuleList([
            LocalAttentionModule(self.group_size, local_hidden)
            for _ in range(n_groups)
        ])

        if self.remaining > 0:
            self.remaining_attention = LocalAttentionModule(self.remaining, local_hidden)

        self.global_attention = GlobalAttentionModule(input_dim, global_hidden)

        self.fusion_gate = nn.Sequential(
            nn.Linear(input_dim * 2, input_dim),
            nn.BatchNorm1d(input_dim),
            nn.Sigmoid()
        )

        self.sparsemax = SparseMax(dim=-1)

    def forward(self, x, prior=None):
        local_outputs = []

        for i in range(self.n_groups):
            start_idx = i * self.group_size
            end_idx = start_idx + self.group_size
            group_features = x[:, start_idx:end_idx]
            local_att = self.local_attentions[i](group_features)
            local_outputs.append(local_att)

        if self.remaining > 0:
            remaining_features = x[:, -self.remaining:]
            remaining_att = self.remaining_attention(remaining_features)
            local_outputs.append(remaining_att)

        local_attention = torch.cat(local_outputs, dim=1)
        global_attention = self.global_attention(x)

        combined = torch.cat([local_attention, global_attention], dim=1)
        fusion_weights = self.fusion_gate(combined)

        attention = fusion_weights * local_attention + (1 - fusion_weights) * global_attention

        if prior is not None:
            attention = attention * prior

        attention_weights = self.sparsemax(attention)
        attended_features = x * attention_weights

        return attended_features, attention_weights


class HierarchicalTabNet(nn.Module):
    def __init__(self, input_dim, output_dim, n_d=32, n_a=32, n_steps=3,
                 n_groups=4, gamma=1.3):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.n_groups = n_groups
        self.gamma = gamma

        self.bn = nn.BatchNorm1d(input_dim)

        self.attention_layers = nn.ModuleList([
            HierarchicalAttentionLayer(
                input_dim=input_dim,
                n_groups=n_groups,
                local_hidden=n_a,
                global_hidden=n_a * 2
            ) for _ in range(n_steps)
        ])

        self.feature_transformers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, n_d + n_a),
                nn.BatchNorm1d(n_d + n_a),
                nn.ReLU()
            ) for _ in range(n_steps)
        ])

        self.final_layer = nn.Linear(n_d * n_steps, output_dim)

    def forward(self, x):
        x = self.bn(x)
        decision_features = []
        prior = torch.ones_like(x)
        self.attention_weights_list = []

        for step in range(self.n_steps):
            attended_features, attention_weights = self.attention_layers[step](x, prior)
            self.attention_weights_list.append(attention_weights)

            transformed = self.feature_transformers[step](attended_features)
            d_features = transformed[:, :self.n_d]
            a_features = transformed[:, self.n_d:]

            decision_features.append(d_features)
            prior = prior * (self.gamma - attention_weights)

        aggregated = torch.cat(decision_features, dim=1)
        logits = self.final_layer(aggregated)

        return logits

    def get_feature_importance(self):
        if not hasattr(self, 'attention_weights_list') or len(self.attention_weights_list) == 0:
            return None
        avg_attention = torch.stack(self.attention_weights_list).mean(dim=0)
        feature_importance = avg_attention.mean(dim=0).detach().cpu().numpy()
        return feature_importance


class TabNetDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class HierarchicalTabNetClassifier:
    def __init__(self, input_dim, output_dim, n_d=32, n_a=32, n_steps=3,
                 n_groups=4, gamma=1.3, lr=0.02, device='cuda'):
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.device = device

        self.model = HierarchicalTabNet(
            input_dim=input_dim,
            output_dim=output_dim,
            n_d=n_d,
            n_a=n_a,
            n_steps=n_steps,
            n_groups=n_groups,
            gamma=gamma
        ).to(device)

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=10, gamma=0.9
        )
        self.criterion = nn.CrossEntropyLoss()

        self.history = {
            'loss': [],
            'train_acc': [],
            'val_acc': []
        }

    def fit(self, X_train, y_train, eval_set=None, eval_name=None, eval_metric=None,
            max_epochs=100, batch_size=256, patience=20, verbose=10, **kwargs):
        train_dataset = TabNetDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        if eval_set is not None and len(eval_set) > 0:
            X_val, y_val = eval_set[0]
            val_dataset = TabNetDataset(X_val, y_val)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        else:
            val_loader = None

        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(max_epochs):
            self.model.train()
            train_loss = 0.0
            train_correct = 0
            train_total = 0

            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)

                self.optimizer.zero_grad()
                logits = self.model(batch_X)
                loss = self.criterion(logits, batch_y)
                loss.backward()
                self.optimizer.step()

                train_loss += loss.item() * batch_X.size(0)
                _, predicted = torch.max(logits, 1)
                train_correct += (predicted == batch_y).sum().item()
                train_total += batch_y.size(0)

            train_loss /= train_total
            train_acc = train_correct / train_total

            if val_loader is not None:
                self.model.eval()
                val_loss = 0.0
                val_correct = 0
                val_total = 0

                with torch.no_grad():
                    for batch_X, batch_y in val_loader:
                        batch_X = batch_X.to(self.device)
                        batch_y = batch_y.to(self.device)

                        logits = self.model(batch_X)
                        loss = self.criterion(logits, batch_y)

                        val_loss += loss.item() * batch_X.size(0)
                        _, predicted = torch.max(logits, 1)
                        val_correct += (predicted == batch_y).sum().item()
                        val_total += batch_y.size(0)

                val_loss /= val_total
                val_acc = val_correct / val_total

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        break
            else:
                val_loss = 0.0
                val_acc = 0.0

            self.history['loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)

            self.scheduler.step()

            if verbose > 0 and (epoch + 1) % verbose == 0:
                if val_loader is not None:
                    print(f"epoch {epoch}: train_loss={train_loss:.5f}, valid_accuracy={val_acc:.5f}")
                else:
                    print(f"epoch {epoch}: train_loss={train_loss:.5f}")

    def predict(self, X):
        self.model.eval()
        X_tensor = torch.FloatTensor(X).to(self.device)

        with torch.no_grad():
            logits = self.model(X_tensor)
            _, predictions = torch.max(logits, 1)

        return predictions.cpu().numpy()

    def predict_proba(self, X):
        self.model.eval()
        X_tensor = torch.FloatTensor(X).to(self.device)

        with torch.no_grad():
            logits = self.model(X_tensor)
            probabilities = F.softmax(logits, dim=1)

        return probabilities.cpu().numpy()

    @property
    def feature_importances_(self):
        if hasattr(self.model, 'attention_weights_list') and len(self.model.attention_weights_list) > 0:
            avg_attention = torch.stack(self.model.attention_weights_list).mean(dim=0)
            return avg_attention.mean(dim=0).detach().cpu().numpy()
        else:
            return np.ones(self.input_dim) / self.input_dim


if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
    torch.cuda.set_device(0)
    device = 'cuda'
else:
    device = 'cpu'

print(f"\ndevice: {device.upper()}")


df = pd.read_csv('liver_cirrhosis.csv')
target_col = 'Stage'
df = df.dropna(subset=[target_col])

leakage_features = [
    'N_Days',
    'Status',
]

removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)


missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} features with high missing rate have been removed")

exclude_kw = ['stage', 'unnamed']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)


X = df[feature_cols].copy()
y = df[target_col].copy()

for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

for i, label in enumerate(le_target.classes_):
    print(f"  Stage {label} → {i}")

print(f"Feature matrix: {X.shape}")

K_FEATURES = 15
selector = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(X, y, random_state=42),
    k=min(K_FEATURES, len(feature_cols))
)
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (score: {score:.4f})")

X_selected = X[selected_features].copy()


class_counts = pd.Series(y_encoded).value_counts()
print("\nNumber of samples per stage:")
for stage_idx, count in class_counts.items():
    stage_name = le_target.classes_[stage_idx]
    print(f"  Stage {stage_name} ({stage_idx}): {count}")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining set distribution (before SMOTE):")
for s, c in pd.Series(y_train).value_counts().sort_index().items():
    print(f"  Stage {le_target.classes_[s]} ({s}): {c}")


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

min_class_samples = pd.Series(y_train).value_counts().min()
k_neighbors = min(5, min_class_samples - 1)
print(f"Minimum class size: {min_class_samples}, using k_neighbors={k_neighbors}")

if k_neighbors >= 1:
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
    X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
else:
    print("Too few samples for SMOTE, using original training set")
    X_train_bal, y_train_bal = X_train_scaled, y_train

for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
    print(f"  Stage {le_target.classes_[s]} ({s}): {c}")


tabnet_model = HierarchicalTabNetClassifier(
    input_dim=X_train_bal.shape[1],
    output_dim=len(np.unique(y_train_bal)),
    n_d=32,
    n_a=32,
    n_steps=3,
    n_groups=4,
    gamma=1.3,
    lr=0.02,
    device=device
)

print("\n" + "─" * 80)
print("Training Hierarchical Attention TabNet...")

、

start_time = time.time()

tabnet_model.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test_scaled, y_test)],
    eval_name=['validation'],
    eval_metric=['accuracy'],
    max_epochs=100,
    patience=20,
    batch_size=256,
    virtual_batch_size=128,
    drop_last=False,
    verbose=10
)

training_time = time.time() - start_time

print(f"TabNet training completed")
print(f"Total training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Actual training epochs: {len(tabnet_model.history['loss'])}")


eval_set = [(X_train_bal, y_train_bal), (X_test_scaled, y_test)]
eval_names = ['train', 'validation']

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=5,
    subsample=1.0,
    colsample_bytree=1.0,
    objective='multi:softprob',
    num_class=len(np.unique(y_train_bal)),
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)


print("Training XGBoost...")

start_time_xgb = time.time()

xgb_model.fit(
    X_train_bal, y_train_bal,
    eval_set=eval_set,
    verbose=10
)

xgb_training_time = time.time() - start_time_xgb

print(f"XGBoost training completed")
print(f"Training time: {xgb_training_time:.2f} seconds ({xgb_training_time/60:.2f} minutes)")

evals_result = xgb_model.evals_result()
train_logloss = evals_result['validation_0']['mlogloss']
val_logloss = evals_result['validation_1']['mlogloss']

best_iteration = np.argmin(val_logloss) + 1


tabnet_train_proba = tabnet_model.predict_proba(X_train_bal)
xgb_train_proba = xgb_model.predict_proba(X_train_bal)

print(f"  TabNet output shape: {tabnet_train_proba.shape}")
print(f"  XGBoost output shape: {xgb_train_proba.shape}")

meta_train_features = np.hstack([tabnet_train_proba, xgb_train_proba])
print(f"  Meta-learner input shape: {meta_train_features.shape}")

meta_learner = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs',
    verbose=1
)

start_time_meta = time.time()

meta_learner.fit(meta_train_features, y_train_bal)

meta_training_time = time.time() - start_time_meta

print(f"Meta-learner training completed")
print(f"  Training time: {meta_training_time:.2f} seconds")
print(f"  Number of iterations: {meta_learner.n_iter_[0]}")


from sklearn.base import BaseEstimator, ClassifierMixin

class StackedEnsemble(BaseEstimator, ClassifierMixin):

    def __init__(self, device='cpu', random_state=42):
        self.device = device
        self.random_state = random_state

    def fit(self, X, y):

        self.tabnet = HierarchicalTabNetClassifier(
            input_dim=X.shape[1] if hasattr(X, 'shape') else len(X[0]),
            output_dim=len(np.unique(y)),
            n_d=32, n_a=32, n_steps=3, n_groups=4, gamma=1.3,
            lr=0.02,
            device=self.device
        )
        self.tabnet.fit(
            X.values if hasattr(X, 'values') else X, y,
            max_epochs=100, patience=20, batch_size=256,
            virtual_batch_size=128, verbose=0
        )

        self.xgb = xgb.XGBClassifier(
            n_estimators=200, learning_rate=0.01, max_depth=5,
            subsample=1.0, objective='multi:softprob',
            num_class=len(np.unique(y)),
            random_state=self.random_state, verbosity=0
        )
        self.xgb.fit(X, y)

        X_arr = X.values if hasattr(X, 'values') else X
        tabnet_proba = self.tabnet.predict_proba(X_arr)
        xgb_proba = self.xgb.predict_proba(X)
        meta_features = np.hstack([tabnet_proba, xgb_proba])

        self.meta = LogisticRegression(
            C=1.0, max_iter=1000, random_state=self.random_state,
            multi_class='multinomial', solver='lbfgs', verbose=0
        )
        self.meta.fit(meta_features, y)

        return self

    def predict(self, X):
        X_arr = X.values if hasattr(X, 'values') else X
        tabnet_proba = self.tabnet.predict_proba(X_arr)
        xgb_proba = self.xgb.predict_proba(X)
        meta_features = np.hstack([tabnet_proba, xgb_proba])
        return self.meta.predict(meta_features)


ensemble_cv = StackedEnsemble(device=device, random_state=42)

cv_start_time = time.time()
cv_scores = cross_val_score(
    ensemble_cv, X_selected, y_encoded,
    cv=5, scoring='accuracy', n_jobs=1
)
cv_time = time.time() - cv_start_time

for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i:2d}: {score:.4f}")
print(f"\n{'='*40}")
print(f"  Average accuracy: {cv_scores.mean():.4f}")
print(f"  Standard deviation: {cv_scores.std():.4f}")
print(f"  Confidence interval: [{cv_scores.mean()-1.96*cv_scores.std():.4f}, {cv_scores.mean()+1.96*cv_scores.std():.4f}]")


xgb_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nXGBoost:")
print(xgb_importance.to_string(index=False))

_ = tabnet_model.predict_proba(X_test_scaled[:1])
tabnet_feature_importance = tabnet_model.feature_importances_

tabnet_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': tabnet_feature_importance
}).sort_values('importance', ascending=False)

print("TabNet:")
print(tabnet_importance.to_string(index=False))


tabnet_train_test_proba = tabnet_model.predict_proba(X_train_scaled)
xgb_train_test_proba = xgb_model.predict_proba(X_train_scaled)
meta_train_test_features = np.hstack([tabnet_train_test_proba, xgb_train_test_proba])
y_train_pred = meta_learner.predict(meta_train_test_features)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_recall = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)

tabnet_test_proba = tabnet_model.predict_proba(X_test_scaled)
xgb_test_proba = xgb_model.predict_proba(X_test_scaled)
meta_test_features = np.hstack([tabnet_test_proba, xgb_test_proba])
y_test_pred = meta_learner.predict(meta_test_features)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)


print("\n" + "=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy-test_accuracy):>15.4f}")
print(f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision-test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall-test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print("=" * 70)

print("=" * 70)
print(classification_report(
    y_test, y_test_pred,
    target_names=[f"Stage {c}" for c in le_target.classes_],
    zero_division=0
))

